In [2]:
from demo_flow import call_llm, flow

with open(r"prompts\v2.md", encoding="utf-8") as f:
    sys = f.read()

event_stream = []
action_taken = False
no_response_counter = 0
res = ""
while True:
    if not action_taken:
        user_msg = input(res)
        if user_msg == "q":
            break
        event_stream.append(f"[User: {user_msg}]")
    agent_output = await call_llm(sys, flow, event_stream, "gpt-4o")
    if agent_output is None:
        break
    if not agent_output.response:
        no_response_counter += 1
        # if no_response_counter >= 3:
        #     event_stream.append("[SYSTEM: You have not responded to the user for three consecutive actions, give them a progress cue]")
    else:
        # reset
        res = agent_output.response
        no_response_counter = 0
    action_taken = bool(agent_output.action) and any(v for k, v in agent_output.action)


<event_stream>
[User: hi, wanted to raise a ticket for the cracks in my kitchen's ceiling]
</event_stream>

<agent_script>
Current Path: Call Start

Node: Call Start
Is Terminal Node?: False
Instructions: Greet the user, Inroduce yourself, then Learn what kind of service does the user need, raise a repair ticket, or update an existing one.
---

Service Type (Radio Field)
( ) Raise repair ticket
( ) Update existing ticket
</agent_script>
ParsedChatCompletionMessage[call_llm.<locals>.AgentOutput](content='{"thoughts":"The user wants to raise a ticket for the cracks in their kitchen\'s ceiling, which means they need to raise a repair ticket. I can select this option to proceed.","response":null,"action":{"field_label":"Service Type","value":"Raise repair ticket"}}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None, parsed=AgentOutput(thoughts="The user wants to raise a ticket for the cracks in their kitchen's ceiling, which means they need to

In [1]:
from demo_flow import flow

In [3]:
flow.current_action_model()

typing.Union[wizard.FillTenantName, wizard.FillTenantAddress, wizard.GoBack, wizard.GoNext, wizard.GoToNode]

In [ ]:
from typing import Union

Union[*[], Union[str, float]]

typing.Union[str, float]

In [ ]:
flow.current_action_model().model_json_schema()

In [1]:
from pydantic import BaseModel

class Test(BaseModel):
    b: str
    a: str

test = Test(a="hello", b="bye")

In [4]:
test.__class__.__name__

'Test'

In [ ]:
for a in test:
    print(a)

In [3]:
from pydantic import BaseModel, create_model, Field

class Dog(BaseModel):
    "A nice lil doggo called lilo"
    bark: int = Field(..., description="bark power")
    name: str

class Cat(BaseModel):
    "cute kitty called bobby"
    meow: int = Field(..., description="meow power")
    name: str


class PetOwner(BaseModel):
    pet: Dog | Cat = Field(..., description="pet of choice")

In [4]:
Dog = create_model(
    "Dog",
    bark = (int, Field(..., description="test desc")),
    test="test",
    __doc__="hello world"

)

PydanticUserError: A non-annotated attribute was detected: `test = 'test'`. All model fields require a type annotation; if `test` is not meant to be a field, you may be able to resolve this error by annotating it as a `ClassVar` or updating `model_config['ignored_types']`.

For further information visit https://errors.pydantic.dev/2.10/u/model-field-missing-annotation

In [ ]:
Dog.model_json_schema()

{'description': 'hello world',
 'properties': {'bark': {'description': 'test desc',
   'title': 'Bark',
   'type': 'integer'}},
 'required': ['bark'],
 'title': 'Dog',
 'type': 'object'}

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

import openai

client = openai.AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])
res = await client.beta.chat.completions.parse(
                model="gpt-4o",
                messages=[
                    {
                        "role": "user",
                        "content": "choose a pet",
                    },
                ],
                response_format=PetOwner,
            )

In [ ]:
PetOwner.model_json_schema()

{'$defs': {'Cat': {'description': 'cute kitty called bobby',
   'properties': {'meow': {'description': 'meow power',
     'title': 'Meow',
     'type': 'integer'},
    'name': {'title': 'Name', 'type': 'string'}},
   'required': ['meow', 'name'],
   'title': 'Cat',
   'type': 'object'},
  'Dog': {'description': 'A nice lil doggo called lilo',
   'properties': {'bark': {'description': 'bark power',
     'title': 'Bark',
     'type': 'integer'},
    'name': {'title': 'Name', 'type': 'string'}},
   'required': ['bark', 'name'],
   'title': 'Dog',
   'type': 'object'}},
 'properties': {'pet': {'anyOf': [{'$ref': '#/$defs/Dog'},
    {'$ref': '#/$defs/Cat'}],
   'description': 'pet of choice',
   'title': 'Pet'}},
 'required': ['pet'],
 'title': 'PetOwner',
 'type': 'object'}

In [ ]:
res.choices[0].message.parsed.model_dump_json()

'{"pet":{"bark":5,"name":"Lilo"}}'

In [ ]:
flow.ctx